In [11]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import httpx

load_dotenv()

client = Anthropic(http_client=httpx.Client(verify=False))
model = "claude-haiku-4-5"

In [12]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [13]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [15]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [18]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt=f"""
Please solve the following task:
{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [19]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the results"""
    output = run_prompt(test_case)

    # TODO -Grading
    score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [20]:
def run_eval(dataset):
    """Loads the dataset and calls run_testa_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    return results

In [21]:
with open("dataset.json") as f:
    dataset = json.load(f)

results = run_eval(dataset)


In [22]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validation Regex\n\nHere's a comprehensive solution:\n\n## Simple Regex Pattern\n\n```regex\n^[a-z0-9]([a-z0-9.-]{1,61}[a-z0-9])?$\n```\n\n## Detailed Explanation\n\nLet me break down what this pattern does:\n\n| Component | Meaning |\n|-----------|---------|\n| `^` | Start of string |\n| `[a-z0-9]` | First character: lowercase letter or digit |\n| `(...)` | Optional middle section (for 3-63 char requirement) |\n| `[a-z0-9.-]{1,61}` | 1-61 middle characters: lowercase, digits, hyphens, periods |\n| `[a-z0-9]` | Last character: lowercase letter or digit |\n| `?` | Middle section is optional (handles 2-char case, though min is 3) |\n| `$` | End of string |\n\n## Implementation Examples\n\n### JavaScript\n```javascript\nconst s3BucketRegex = /^[a-z0-9]([a-z0-9.-]{1,61}[a-z0-9])?$/;\n\n// Test cases\nconsole.log(s3BucketRegex.test(\"my-bucket\"));        // true\nconsole.log(s3BucketRegex.test(\"my-bucket-2\"));      // true\nconsole.log(s3BucketRe